# Grouped leakage protocol if verified relationships exist

Creates new grouped folds only after human/source verification. It does not refit a neural model.

Run the single code cell below with the **Python (skin)** kernel.

In [ ]:
from pathlib import Path
import sys

PACKAGE_DIR_INPUT = Path(r"D:\AIUB\Neural\A_Comprehensive_Framework_for_Auditing_Demographic_Bias_in_Synthetic_Skin_Disease_Datasets\revision_package")
if not PACKAGE_DIR_INPUT.is_dir():
    raise FileNotFoundError(f"revision_package not found: {PACKAGE_DIR_INPUT}")
if str(PACKAGE_DIR_INPUT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR_INPUT))

import shutil
import pandas as pd

from notebook_config import RUN_ROOT
from notebook_support import ensure_core_outputs, run_once

audit_metadata, _ = ensure_core_outputs(RUN_ROOT)
raw_candidates = RUN_ROOT / "data_audit" / "duplicate_candidates_for_review.csv"
reviewed_pairs = RUN_ROOT / "reviewed_duplicate_pairs.csv"
if not reviewed_pairs.exists():
    shutil.copy2(raw_candidates, reviewed_pairs)
    print(f"Created review copy: {reviewed_pairs}")
    print("Open it, inspect every candidate using source/visual evidence, fill verified_related with yes or no, save, then rerun this cell.")
else:
    pairs = pd.read_csv(reviewed_pairs).fillna("")
    status = pairs["verified_related"].astype(str).str.strip().str.lower() if len(pairs) else pd.Series(dtype=str)
    unresolved = int((~status.isin(["yes", "no"])).sum())
    metadata = pd.read_csv(audit_metadata)
    identity_fields = [field for field in ["patient_id", "case_id", "parent_id", "acquisition_session"] if field in metadata]
    has_verified_ids = any(
        metadata[field].notna().any() and
        (~metadata[field].astype(str).str.strip().str.lower().isin(["", "unknown", "missing", "nan"])).any()
        for field in identity_fields
    )
    yes_pairs = int(status.eq("yes").sum())
    if unresolved:
        print(f"STOP: {unresolved} candidate rows still need yes/no review in {reviewed_pairs}")
    elif yes_pairs == 0 and not has_verified_ids:
        print("All candidates were reviewed, but no verified related pairs or patient/case identifiers exist.")
        print("Do not claim patient independence; a grouped reanalysis is not identifiable from the available metadata.")
    else:
        grouped_dir = run_once(
            "06_make_grouped_splits.py",
            ["--metadata", str(audit_metadata), "--reviewed-pairs", str(reviewed_pairs)],
            RUN_ROOT / "grouped_protocol",
            ["grouped_fold_manifest.csv", "grouped_fold_counts.csv", "metadata_with_clusters.csv", "protocol.json"],
        )
        print(pd.read_csv(grouped_dir / "grouped_fold_counts.csv").to_string(index=False))
        print("\nThese are NEW folds. Old checkpoints cannot score them; refitting would be required.")
